# User Coherence Evaluation on Foundry Agent Traces

Evaluates user coherence (derail detection) on multi-turn conversations fetched from Azure AI Foundry agents:
- **legal-contract-review-sim** (20 conversations)
- **airline-customer-service-sim** (30 conversations)

Each conversation is evaluated independently, producing per-step derail scores and an aggregate mean score.

In [10]:
import json
import os
import asyncio
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

# All files (datasets, .env, results) live in the same directory as this notebook
load_dotenv()

# Import evaluator and trace normalizer
from azure.ai.evaluation._evaluators._user_coherence._user_coherence import (
    UserCoherenceEvaluator,
    normalize_trace_messages,
)

# Load datasets (produced by fetch_agent_conversations.py in the same folder)
AGENTS = os.environ.get("AGENTS", "legal-contract-review-sim,airline-customer-service-sim").split(",")

datasets = {}
for agent_name in AGENTS:
    filename = f"{agent_name}_conversations.json"
    if not Path(filename).exists():
        print(f"WARNING: {filename} not found — run fetch_agent_conversations.py first")
        continue
    with open(filename, encoding="utf-8") as f:
        datasets[agent_name] = json.load(f)
    print(f"Loaded {len(datasets[agent_name])} conversations for {agent_name}")

Loaded 50 conversations for legal-contract-review-sim
Loaded 50 conversations for airline-customer-service-sim


## Preview: Normalize Trace Messages

The raw trace messages from Foundry use `output_text`/`input_text` content types and include extra metadata. `normalize_trace_messages()` converts them to simple `{"role": ..., "content": ...}` dicts.

In [11]:
# Preview normalization on the first conversation of each agent
# NOTE: Foundry conversations are already in chronological order.
for agent_name, convs in datasets.items():
    raw_msgs = convs[0]["messages"]
    normalized = normalize_trace_messages(raw_msgs)
    print(f"\n=== {agent_name} (conversation {convs[0]['conversation_id']}) ===")
    print(f"Raw messages: {len(raw_msgs)}, Normalized: {len(normalized)}")
    for msg in normalized:
        print(f"  [{msg['role']}] {msg['content']}...")


=== legal-contract-review-sim (conversation conv_0842a7cbecd7e0cf00uPEBlTji4cRqO5iK5q7btmr4ZP5UGqVG) ===
Raw messages: 10, Normalized: 10
  [user] Hey, this is Olivia Mensah. Quick one — is the limitation-of-liability clause in this SaaS agreement ok? I need to sign today, so I really need a fast answer....
  [assistant] Hi Olivia! To properly assess the limitation-of-liability clause for your SaaS agreement, I need to know:

1. Which contract you’re referencing (contract_id or the agreement’s name).
2. What jurisdiction governs the contract (as some rules vary).
3. Any company policy specifics (for example, are uncapped liabilities always unacceptable?).

To review quickly, I’ll:
- Retrieve contract metadata for context,
- Locate and read the actual limitation-of-liability clause (not just the heading or search hits),
- Check if the clause aligns with your company policy and jurisdictional rules,
- Explain any material risks in plain language and propose fixes if needed.

Please prov

## Initialize UserCoherenceEvaluator

Configure the evaluator with an Azure OpenAI model deployment.

In [12]:
# Read Azure OpenAI config from environment (loaded from .env above)
AZURE_OPENAI_ENDPOINT = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
AZURE_OPENAI_API_KEY = os.environ.get("AZURE_OPENAI_API_KEY", "")
AZURE_OPENAI_API_VERSION = os.environ.get("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")
AZURE_OPENAI_DEPLOYMENT = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4.1")

print(f"Endpoint: {AZURE_OPENAI_ENDPOINT}")
print(f"Deployment: {AZURE_OPENAI_DEPLOYMENT}")
print(f"API version: {AZURE_OPENAI_API_VERSION}")
print(f"API key: {'***' + AZURE_OPENAI_API_KEY[-4:] if AZURE_OPENAI_API_KEY else '(not set)'}")

Endpoint: https://eval-cairo-ai-foundry-eus.cognitiveservices.azure.com/
Deployment: gpt-4.1
API version: 2024-12-01-preview
API key: ***hlhF


In [13]:
from azure.ai.evaluation import AzureOpenAIModelConfiguration
from azure.identity import DefaultAzureCredential

model_config = AzureOpenAIModelConfiguration(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    azure_deployment=AZURE_OPENAI_DEPLOYMENT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
)

credential = DefaultAzureCredential()

evaluator = UserCoherenceEvaluator(model_config=model_config, credential=credential)
print("UserCoherenceEvaluator initialized")

UserCoherenceEvaluator initialized


## Run Evaluation Per Conversation

For each agent, evaluate every conversation independently and collect results.

In [14]:
import time

all_results = {}  # agent_name -> list of result dicts

for agent_name, convs in datasets.items():
    print(f"\n{'='*80}")
    print(f"Evaluating: {agent_name} ({len(convs)} conversations)")
    print(f"{'='*80}")

    agent_results = []

    for i, conv in enumerate(convs):
        conv_id = conv["conversation_id"]
        raw_messages = conv["messages"]

        # Normalize (messages are already in chronological order)
        messages = normalize_trace_messages(raw_messages)

        if len(messages) < 2:
            print(f"  [{i+1}/{len(convs)}] {conv_id[:30]}... SKIPPED (only {len(messages)} messages)")
            continue

        print(f"  [{i+1}/{len(convs)}] {conv_id[:30]}... ({len(messages)} messages)", end="")

        try:
            # Evaluate using the conversation input format
            result = evaluator(conversation={"messages": messages})
            score = result.get("user_coherence", float("nan"))
            per_step = result.get("user_coherence_per_step", [])
            print(f" -> score={score:.2f} ({len(per_step)} steps)")

            agent_results.append({
                "conversation_id": conv_id,
                "agent_name": agent_name,
                "num_messages": len(messages),
                "user_coherence": score,
                "per_step_scores": per_step,
                "total_tokens": result.get("user_coherence_total_tokens", 0),
                "model": result.get("user_coherence_model", ""),
            })
        except Exception as e:
            print(f" -> ERROR: {e}")
            agent_results.append({
                "conversation_id": conv_id,
                "agent_name": agent_name,
                "num_messages": len(messages),
                "user_coherence": float("nan"),
                "error": str(e),
            })

        # Small delay to avoid rate limiting
        time.sleep(0.5)

    all_results[agent_name] = agent_results
    print(f"\n  Completed {len(agent_results)} evaluations for {agent_name}")


Evaluating: legal-contract-review-sim (50 conversations)
  [1/50] conv_0842a7cbecd7e0cf00uPEBlTj... (10 messages) -> score=2.00 (5 steps)
  [2/50] conv_c43ff32bd5c4f0f800gDkl0Wz... (10 messages) -> score=2.00 (5 steps)
  [3/50] conv_8d4b2d823e8a1d2100HK9q8gZ... (10 messages) -> score=2.00 (5 steps)
  [4/50] conv_a52d0cd1505aa0920098fmBAk... (10 messages) -> score=2.00 (5 steps)
  [5/50] conv_a2650e815ac77d7d00sjZYSMb... (10 messages) -> score=2.00 (5 steps)
  [6/50] conv_5844bba372c6c36f00wFpFAQQ... (11 messages) -> score=2.00 (5 steps)
  [7/50] conv_3b52bf7f44771fe400lXmmUqE... (9 messages) -> score=2.00 (4 steps)
  [8/50] conv_7d4e3ef070b0583c00bekHSCL... (8 messages) -> score=2.00 (4 steps)
  [9/50] conv_7309955b0d21b1d400p14ZoDT... (8 messages) -> score=2.00 (4 steps)
  [10/50] conv_f064f710eb0666b800Iv1UBib... (10 messages) -> score=2.00 (5 steps)
  [11/50] conv_4dd34510dd1753f6003QwF4Ot... (11 messages) -> score=2.00 (5 steps)
  [12/50] conv_df1c9b1bbf35a8cc00vsHQm0s... (10 mess

## Results Summary

Aggregate results per agent and display a summary table.

In [18]:
import math

# Build summary DataFrames
for agent_name, results in all_results.items():
    print(f"\n{'='*60}")
    print(f"Agent: {agent_name}")
    print(f"{'='*60}")

    df = pd.DataFrame([
        {
            "conversation_id": r["conversation_id"][:30] + "...",
            "messages": r["num_messages"],
            "user_coherence": r["user_coherence"],
            "tokens": r.get("total_tokens", 0),
        }
        for r in results
    ])

    display(df)

    valid_scores = [r["user_coherence"] for r in results if not math.isnan(r["user_coherence"])]
    if valid_scores:
        print(f"\n  Conversations evaluated: {len(valid_scores)}")
        print(f"  Mean user_coherence: {sum(valid_scores)/len(valid_scores):.3f}")
        print(f"  Min: {min(valid_scores):.2f}, Max: {max(valid_scores):.2f}")
        print(f"  Std: {pd.Series(valid_scores).std():.3f}")
    else:
        print("  No valid scores")


Agent: legal-contract-review-sim


,conversation_id,messages,user_coherence,tokens
0,conv_0842a7cbecd7e0cf00uPEBlTj...,10,2.0,5321
1,conv_c43ff32bd5c4f0f800gDkl0Wz...,10,2.0,4753
2,conv_8d4b2d823e8a1d2100HK9q8gZ...,10,2.0,4656
3,conv_a52d0cd1505aa0920098fmBAk...,10,2.0,4430
4,conv_a2650e815ac77d7d00sjZYSMb...,10,2.0,5358
5,conv_5844bba372c6c36f00wFpFAQQ...,11,2.0,5268
6,conv_3b52bf7f44771fe400lXmmUqE...,9,2.0,4592
7,conv_7d4e3ef070b0583c00bekHSCL...,8,2.0,5666
8,conv_7309955b0d21b1d400p14ZoDT...,8,2.0,4749
9,conv_f064f710eb0666b800Iv1UBib...,10,2.0,5111



  Conversations evaluated: 50
  Mean user_coherence: 2.000
  Min: 2.00, Max: 2.00
  Std: 0.000

Agent: airline-customer-service-sim


,conversation_id,messages,user_coherence,tokens
0,conv_b4dd30d64e6b0dc300V6yB6BN...,10,2.0,3743
1,conv_06b3877fbbeca15200ZMTAGL9...,15,2.0,4584
2,conv_84f6392575738eba00kqxeUlq...,14,2.0,4780
3,conv_4b0a422287a2e5cb00PtEbvVz...,17,2.0,5334
4,conv_c2385265a03d6fba00eGzsnQB...,14,2.0,4756
5,conv_eb3c9f2328e4671d00RC5hJ04...,15,2.0,4586
6,conv_ce490d3bf1b340e00071ZcwPo...,13,2.0,4443
7,conv_6341b1140363451400dpZZEIX...,10,2.0,3779
8,conv_54d29dbaf3d92b3800UlavsFp...,14,2.0,4452
9,conv_064412c823b1ad0b00gCO1sEj...,17,2.0,4863



  Conversations evaluated: 50
  Mean user_coherence: 2.000
  Min: 2.00, Max: 2.00
  Std: 0.000


## Per-Step Score Distribution

Visualize the distribution of per-step derail scores across all conversations for each agent.

In [16]:
from collections import Counter

for agent_name, results in all_results.items():
    print(f"\n{'='*60}")
    print(f"Agent: {agent_name} — Per-Step Score Distribution")
    print(f"{'='*60}")

    score_counts = Counter()
    total_steps = 0
    derailed_conversations = []

    for r in results:
        per_step = r.get("per_step_scores", [])
        for step in per_step:
            s = step.get("derail_score")
            if s is not None:
                score_counts[s] += 1
                total_steps += 1

        # Flag conversations with any score=0 (derailed)
        zeros = [step for step in per_step if step.get("derail_score") == 0]
        if zeros:
            derailed_conversations.append({
                "conversation_id": r["conversation_id"][:30] + "...",
                "derailed_steps": [
                    {"step": z.get("conversation_step"), "explanation": z.get("explanation", "")}
                    for z in zeros
                ],
            })

    print(f"\n  Total scored steps: {total_steps}")
    for score in [2, 1, 0]:
        count = score_counts.get(score, 0)
        pct = (count / total_steps * 100) if total_steps else 0
        bar = "█" * int(pct / 2)
        label = {2: "Likely/logical", 1: "Weak/possible", 0: "Not logical"}[score]
        print(f"  Score {score} ({label}): {count:3d} ({pct:5.1f}%) {bar}")

    if derailed_conversations:
        print(f"\n  Conversations with derailed steps (score=0):")
        for dc in derailed_conversations:
            print(f"    {dc['conversation_id']}")
            for ds in dc["derailed_steps"]:
                print(f"      Step {ds['step']}: {ds['explanation']}")


Agent: legal-contract-review-sim — Per-Step Score Distribution

  Total scored steps: 194
  Score 2 (Likely/logical): 194 (100.0%) ██████████████████████████████████████████████████
  Score 1 (Weak/possible):   0 (  0.0%) 
  Score 0 (Not logical):   0 (  0.0%) 

Agent: airline-customer-service-sim — Per-Step Score Distribution

  Total scored steps: 244
  Score 2 (Likely/logical): 244 (100.0%) ██████████████████████████████████████████████████
  Score 1 (Weak/possible):   0 (  0.0%) 
  Score 0 (Not logical):   0 (  0.0%) 


## Save Results

Save the full evaluation results (including per-step details) to JSON files.

In [17]:
for agent_name, results in all_results.items():
    output_file = f"{agent_name}_user_coherence_results.json"
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, default=str)
    print(f"Saved {len(results)} results to {output_file}")

Saved 50 results to legal-contract-review-sim_user_coherence_results.json
Saved 50 results to airline-customer-service-sim_user_coherence_results.json
